一些小总结：
激活函数activate function：sigimoid  tahn  RELU  gener-Relu  softmax

loss function：CrossEntropyLoss交叉熵/二分类交叉熵损失函数BCELoss(Binary Cross Entropy Loss)  MSE均方误差（有缺陷）  
Loss = -log(softmax(outputs)[class_index])

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


# 自定义数据集
class MNISTDataset(Dataset):
    def __init__(self, file_path):
        self.images, self.labels = self._read_file(file_path)

    def _read_file(self, file_path):
        images = []
        labels = []
        with open(file_path, 'r') as f:
            next(f)  # 跳过标题行
            for line in f:
                items = line.strip().split(",")
                images.append([float(x) for x in items[1:]])
                labels.append(int(items[0]))
        return images, labels

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).view(-1)
        image = image / 255.0  # 归一化
        image = (image - 0.1307) / 0.3081  # 标准化
        label = torch.tensor(self.labels[index], dtype=torch.long)
        return image, label

    def __len__(self):
        return len(self.images)

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)

可以看到定义了一个神经网络的类，它继承自nn.Module，我们就可以直接利用PyTorch定义好的Linear和ReLU。在初始化函数里，我们定义了一个nn.Sequential的对象，它会顺序链接内部各个模块。forward方法就是直接调用我们定义的model。

In [ ]:
# 参数设置
batch_size = 64
learning_rate = 0.1
num_epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 数据加载
train_dataset = MNISTDataset(r'F:\VS code project\AI\Learning\PyTorch\MNIST\mnist_train.csv')
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)#训练随机打乱
test_dataset = MNISTDataset(r'F:\VS code project\AI\Learning\PyTorch\MNIST\mnist_test.csv')
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)#检查精度，需要复现数据

# 模型、损失函数、优化器
model = NeuralNetwork().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
'''我们利用PyTorch内部定义的交叉熵损失函数，以及优化器。'''

In [5]:
# 训练过程
model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels) # 计算loss

        optimizer.zero_grad() # 清理梯度
        loss.backward() # 反向传播
        optimizer.step() # 更新参数

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

Epoch 1/10, Loss: 0.4459
Epoch 2/10, Loss: 0.1211
Epoch 3/10, Loss: 0.0821
Epoch 4/10, Loss: 0.0607
Epoch 5/10, Loss: 0.0469
Epoch 6/10, Loss: 0.0386
Epoch 7/10, Loss: 0.0300
Epoch 8/10, Loss: 0.0263
Epoch 9/10, Loss: 0.0218
Epoch 10/10, Loss: 0.0189


In [6]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 97.90%


In [8]:
#完整代码
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


# 自定义数据集
class MNISTDataset(Dataset):
    def __init__(self, file_path):
        self.images, self.labels = self._read_file(file_path)

    def _read_file(self, file_path):
        images = []
        labels = []
        with open(file_path, 'r') as f:
            next(f)  # 跳过标题行
            for line in f:
                items = line.strip().split(",")
                images.append([float(x) for x in items[1:]])
                labels.append(int(items[0]))
        return images, labels

    def __getitem__(self, index):
        image = torch.tensor(self.images[index], dtype=torch.float32).view(-1)
        image = image / 255.0  # 归一化
        image = (image - 0.1307) / 0.3081  # 标准化
        label = torch.tensor(self.labels[index], dtype=torch.long)
        return image, label

    def __len__(self):
        return len(self.images)


# 模型定义
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)


# 参数设置
batch_size = 64
learning_rate = 0.1
num_epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 数据加载
train_dataset = MNISTDataset(r'F:\VS code project\AI\Learning\PyTorch\MNIST\mnist_train.csv')
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)#训练随机打乱
test_dataset = MNISTDataset(r'F:\VS code project\AI\Learning\PyTorch\MNIST\mnist_test.csv')
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)#检查精度，需要复现数据

# 模型、损失函数、优化器
model = NeuralNetwork().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

# 训练过程
model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

# 测试过程
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Epoch 1/10, Loss: 0.4912
Epoch 2/10, Loss: 0.1267
Epoch 3/10, Loss: 0.0843
Epoch 4/10, Loss: 0.0647
Epoch 5/10, Loss: 0.0503
Epoch 6/10, Loss: 0.0402
Epoch 7/10, Loss: 0.0339
Epoch 8/10, Loss: 0.0265
Epoch 9/10, Loss: 0.0222
Epoch 10/10, Loss: 0.0163
Test Accuracy: 98.03%


线性函数如果不构造高阶特征，只能拟合直线，对于折线就无能为力了

折线分为3段，我们可以将它拆成3段分别拟合。

为了让函数在整个值域有定义，我们用直线补齐折线的其他定义域。

最后做一下调整，除了第一个折线外，让其他折线的起始值都为0。这样就得到下边的三个图像。

把原始需要我们拟合的函数称为
f
(
x
)
f(x)，把下边这三个看起来像阶梯的函数叫做阶梯函数，分别为
a
1
(
x
)
,
a
2
(
x
)
,
a
3
(
x
)
a 
1
​
 (x),a 
2
​
 (x),a 
3
​
 (x)。经过上边的拆解和平移后，
f
(
x
)
f(x)在某一点的取值，就等于在这一点
a
1
(
x
)
+
a
2
(
x
)
+
a
3
(
x
)
a 
1
​
 (x)+a 
2
​
 (x)+a 
3
​
 (x)的值。也就是说只要我们能找到拟合阶梯图像的函数，通过把三个阶梯函数加和，我们就可以拟合上边的折线。接下来我们就来寻找合适的阶梯函数。

它对sigmoid函数沿着x轴方向进行了压缩，看着更像一个阶梯函数了。w取负会上下翻转，b取不同值会左右平移

但是有一个问题，就是函数的值域一直是0到1，为了调整图像在y轴上的变化，我们对Sigmoid函数的输出也加上两个参数来进行线性控制，这里我们用大写的
W
,
B
W,B来表示。

这样我们就可以用这三个Sigmoid拟合的阶梯函数的加和来拟合原始的折线了。

我们是用多个带参数的Sigmoid函数拟合了一个折线，但是这和神经网络有什么关系呢？

实际上3个阶梯函数就代表3个隐藏层的神经元，小写的
w
,
b
w,b是隐藏层的权重和偏置。大写的
W
,
B
W,B是输出层的权重和偏置。Sigmoid就是每个神经元的激活函数。这个神经网络只有一个输入x，一个输出y。

上边我们直观解释了用两层的神经网络来拟合一个3段的折线。任意曲线都可以看做是由很多段的折线拼接构成。所以2层的神经网络理论上可以拟合任意函数。

你可能会好奇，如果激活函数选择ReLU该怎么拟合呢？Sigmoid函数或者tanh函数本来长得就像阶梯函数，但是ReLU怎么拟合阶梯函数呢？答案是用两个ReLU函数的和来构成一个阶梯函数。

同样我们给ReLU函数也增加
w
,
b
,
W
,
B
w,b,W,B四个参数：

y
=
W
∗
R
e
L
U
(
w
x
+
b
)
+
B
y=W∗ReLU(wx+b)+B

这就相当于是一个用ReLU作为激活函数的神经元，我们构造两个这样的神经元。

所以用ReLU作为激活函数，也是可以拟合任意函数的。

既然两层的神经网络理论上就可以拟合任意函数了，为什么我们还需要深层神经网络？这是因为深层的神经网络可以逐层提取更高级的特征，正是这种逐层抽象的能力大大提升了神经网络的能力。完成同一个任务，两层的神经网络需要的神经元的数量要指数级大于深度神经网络。

就像上边这张图所示，它展示了利用多层神经网络来进行人脸识别。浅层的神经网络提取的是一些简单特征，比如点、线。中间层利用浅层提取的点和线作为输入，识别的是一些脸部器官，比如眼睛，鼻子，嘴巴。而最终深层的网络输入就是眼睛、鼻子、嘴巴这样的器官，最终输出对人脸的识别结果。深度神经网络就是利用了这种逐层抽象的能力来达到好的效果。 逐层抽象，相当于对一个复杂任务做了分解，每一层都在上一层的基础上进行简单的组合和加工。每一层要学习的内容就简单。模型更容易达到好的效果。有研究论文发现，同等参数量的模型，一个浅而宽的神经网络效果是远不如一个窄而深的神经网络的。当然每一层的神经元也不能太少，至少要能学习到这一层的所有模式才行，这一层是学习的人脸的器官，那必须有神经元分别是识别眼睛、眼镜、眉毛、额头、嘴巴、鼻子、耳朵、脸颊、下巴等等。